### Agent-Lab: Multi-Agent -> Browser

Objective of this notebook is evaluating and adapting a [Multi-Agent Supervisor Architecture](https://langchain-ai.github.io/langgraph/concepts/multi_agent/#supervisor) with coordinator and execution planning steps.

---


In [1]:
%%capture
import json
import os
import nest_asyncio
from IPython.display import Markdown, display
from dotenv import load_dotenv
from notebooks import experiment_utils
from agent_lab.interface.api.messages.schema import MessageRequest


os.chdir("..")
load_dotenv()
nest_asyncio.apply()

# start dependency injection container
container = experiment_utils.bootstrap_container([__name__])

# get checkpointer instance
graph_persistence_factory = container.graph_persistence_factory()
checkpointer = graph_persistence_factory.build_checkpoint_saver()

---
### Anthropic Browser

In [2]:
# Create Workflow
anthropic_agent = experiment_utils.create_anthropic_agent(
    agent_type="coordinator_planner_supervisor", llm_tag="claude-sonnet-5", api_key=os.getenv("ANTHROPIC_API_KEY")
)
anthropic_browser_agent = container.agent_registry().get_agent("coordinator_planner_supervisor")
anthropic_workflow_builder = anthropic_browser_agent.get_workflow_builder(anthropic_agent["id"])
anthropic_workflow = anthropic_workflow_builder.compile(checkpointer=checkpointer)

In [3]:
%%capture

message = MessageRequest(
    message_role="human",
    message_content="Using web browser at your disposal, visit https://en.wikipedia.org/wiki/Mathematical_finance and rewrite the first paragraph in a simpler way that a 12 year old would understand.",
    agent_id=anthropic_agent["id"],
)

inputs = anthropic_browser_agent.get_input_params(message, schema="public")
config = anthropic_browser_agent.get_config(anthropic_agent["id"])
result = anthropic_workflow.invoke(inputs, config)
ai_message_content, workflow_state = anthropic_browser_agent.format_response(result)

In [4]:
display(
    Markdown(
        f"**Execution Plan:**\n```json\n{json.dumps(workflow_state.get("execution_plan"), indent=2)}\n```"
    )
)

**Execution Plan:**
```json
{
  "thought": "The user wants me to visit a specific Wikipedia page using a web browser and then rewrite the first paragraph in simpler language suitable for a 12-year-old. This requires direct interaction with a webpage to extract the content (best done by browser agent since it's a specific URL visit), followed by simplifying the text (reporter can handle the final rewrite/presentation). Since this is a simple two-step task: (1) browse the page to get the first paragraph, (2) rewrite it simply and present as final report.",
  "title": "\u8bbf\u95ee\u7ef4\u57fa\u767e\u79d1\u9875\u9762\u5e76\u7528\u7b80\u5355\u8bed\u8a00\u6539\u5199\u7b2c\u4e00\u6bb5",
  "steps": [
    {
      "agent_name": "browser",
      "title": "\u8bbf\u95ee\u7ef4\u57fa\u767e\u79d1\u9875\u9762\u83b7\u53d6\u7b2c\u4e00\u6bb5\u5185\u5bb9",
      "description": "\u4f7f\u7528\u6d4f\u89c8\u5668\u6253\u5f00 https://en.wikipedia.org/wiki/Mathematical_finance \uff0c\u5b9a\u4f4d\u5e76\u63d0\u53d6\u9875\u9762\u6b63\u6587\u7684\u7b2c\u4e00\u6bb5\u6587\u5b57\u5185\u5bb9\u3002\u8f93\u51fa\uff1a\u8be5\u9875\u9762\u7b2c\u4e00\u6bb5\u7684\u539f\u6587\u6587\u672c\u3002"
    },
    {
      "agent_name": "reporter",
      "title": "\u7528\u7b80\u5355\u8bed\u8a00\u6539\u5199\u5e76\u8f93\u51fa\u6700\u7ec8\u62a5\u544a",
      "description": "\u57fa\u4e8e\u6d4f\u89c8\u5668\u83b7\u53d6\u7684\u7b2c\u4e00\u6bb5\u539f\u6587\uff0c\u5c06\u5176\u6539\u5199\u4e3a\u9002\u540812\u5c81\u5b69\u5b50\u7406\u89e3\u7684\u7b80\u5355\u8bed\u8a00\u7248\u672c\uff0c\u4f7f\u7528\u7b80\u77ed\u53e5\u5b50\u3001\u5e38\u89c1\u8bcd\u6c47\u548c\u6613\u61c2\u7684\u6bd4\u55bb\uff0c\u907f\u514d\u4e13\u4e1a\u672f\u8bed\u3002\u8f93\u51fa\uff1a\u4e00\u4efd\u5305\u542b\u539f\u6587\uff08\u53ef\u9009\uff09\u548c\u7b80\u5316\u7248\u6539\u5199\u5185\u5bb9\u7684\u6700\u7ec8\u62a5\u544a\u3002"
    }
  ]
}
```

In [5]:
display(Markdown(f"**AI Message Content:**\n\n{ai_message_content}"))

**AI Message Content:**

# Mathematical Finance - Simplified for a 12-Year-Old

## Original First Paragraph (from Wikipedia):
> "Mathematical finance, also known as quantitative finance and financial mathematics, is a field of applied mathematics, concerned with mathematical modeling in the financial field."

## Simplified Version:

**Mathematical finance** is a special kind of math that helps people understand money stuff — like how prices change, how much something might be worth in the future, or how risky an investment could be. 

Think of it like using math as a tool to predict and understand things related to money, kind of like how a weather forecaster uses math and science to predict if it will rain tomorrow. Instead of predicting weather, these mathematicians use numbers, formulas, and patterns to help banks, companies, and investors make smarter decisions about money.

It's sometimes also called "quantitative finance" — which is just a fancy way of saying "using numbers and math to study finance (money matters)."

---
### Local Browser

In [6]:
# Create Workflow
local_agent = experiment_utils.create_local_agent(
    agent_type="coordinator_planner_supervisor", llm_tag=os.getenv("SIMULATION_LLM_MODEL"),
    local_endpoint=os.getenv("SIMULATION_LLM_API_BASE")
)
experiment_utils.enable_jev(local_agent["id"], api_key=os.getenv("TYPESAFE_API_KEY"))
experiment_utils.update_language_model_setting(
    local_agent["language_model_id"], "structured_output_method", "function_calling"
)
local_browser_agent = container.agent_registry().get_agent("coordinator_planner_supervisor")
local_workflow_builder = local_browser_agent.get_workflow_builder(local_agent["id"])
local_workflow = local_workflow_builder.compile(checkpointer=checkpointer)

In [7]:
print(f"Model: {os.getenv('SIMULATION_LLM_MODEL')}")

Model: GLM-5.3-Flash-EXL3


In [8]:
%%capture

message = MessageRequest(
    message_role="human",
    message_content="Using web browser at your disposal, visit https://en.wikipedia.org/wiki/Mathematical_finance and rewrite the first paragraph in a simpler way that a 12 year old would understand.",
    agent_id=local_agent["id"],
)

inputs = local_browser_agent.get_input_params(message, schema="public")
config = local_browser_agent.get_config(local_agent["id"])
result = local_workflow.invoke(inputs, config)
ai_message_content, workflow_state = local_browser_agent.format_response(result)

In [9]:
display(
    Markdown(
        f"**Execution Plan:**\n```json\n{json.dumps(workflow_state.get("execution_plan"), indent=2)}\n```"
    )
)

**Execution Plan:**
```json
{
  "steps": [
    {
      "agent_name": "browser",
      "description": "Open https://en.wikipedia.org/wiki/Mathematical_finance and extract the first (lead) paragraph of the article verbatim, so it can be simplified in the next step. Note: This requires direct web page interaction, so the browser agent is used.",
      "title": "Visit the Wikipedia page on Mathematical Finance"
    },
    {
      "agent_name": "reporter",
      "description": "Take the first paragraph extracted from the Wikipedia page and rewrite it in very simple, friendly language that a 12-year-old can easily understand (short sentences, everyday words, relatable examples). Present the original paragraph followed by the simplified rewrite as the final answer.",
      "title": "Rewrite the first paragraph for a 12-year-old"
    }
  ],
  "thought": "The user wants me to visit the Wikipedia page on Mathematical Finance using the web browser, read its first paragraph, and rewrite that paragraph in simple language that a 12-year-old would understand. My plan: first, the browser agent will visit the page and extract the exact text of the first paragraph. Then, the reporter will take that paragraph and rewrite it in simple, kid-friendly language, presenting the final result. No math or coding is needed, so no researcher or coder steps are required.",
  "title": "Simplify the first paragraph of the Wikipedia \"Mathematical finance\" article for a 12-year-old"
}
```

In [10]:
display(Markdown(f"**AI Message Content:**\n\n{ai_message_content}"))

**AI Message Content:**

I successfully retrieved the first paragraph from the Wikipedia page. Here's what I found and the simplified rewrite:

## Original Paragraph (from Wikipedia)

> "Mathematical finance, also known as quantitative finance and financial mathematics, is a field of applied mathematics, concerned with mathematical modeling in the financial field."

## Simplified Version (for a 12-year-old)

> Mathematical finance is a type of math that people use to understand money. Some people call it "quantitative finance" or "financial mathematics," but they all mean the same thing. Instead of just doing math for homework, people in this field use numbers and equations to figure out how money works in the real world — like how banks, stocks, and investments behave.

**Why this works for a 12-year-old:**
- Replaced the formal term "field of applied mathematics" with the relatable idea of "a type of math people use to understand money"
- Explained "mathematical modeling" as "using numbers and equations to figure out how money works in the real world"
- Added a concrete, familiar example (banks, stocks, investments) to make the abstract concept tangible
- Clarified that the different names (quantitative finance, financial mathematics) all mean the same thing